In [0]:
# =====================================================================
# proceso / Orquestador.py  (OPCIONAL - solo referencia didáctica)
#
# Alternativa simple de orquestación vista en clase: encadenar
# dbutils.notebook.run() en un solo notebook. A diferencia del Job de
# CI/CD (creado por .github/workflows/cicd.yml, patrón YML3, task usado
# en producción real), esta forma:
#   - Ejecuta las tareas de forma SECUENCIAL (no permite el paralelismo
#     ingest_superstore / ingest_ecommerce / ingest_calendar).
#   - Requiere rutas de notebook ABSOLUTAS (/Workspace/Users/...), lo
#     que la hace poco portable entre dev y prod.
#
# Se deja aquí solo como referencia; el pipeline que SÍ se despliega
# vía CI/CD es el Job creado por cicd.yml.
# =====================================================================

In [0]:
CATALOGO = "retail_medallion"
STORAGE_NAME = "<STORAGE_ACCOUNT_NAME>"
BASE_PATH = "/Workspace/Repos/<usuario_o_repo>/databricks-app/proceso"

In [0]:
dbutils.notebook.run(
    f"{BASE_PATH}/00_prepamb.py",
    timeout_seconds=300,
    arguments={"sql_file": "PrepAmb/01_prepamb.sql", "catalogo": CATALOGO, "storageName": STORAGE_NAME},
)

In [0]:
dbutils.notebook.run(
    f"{BASE_PATH}/01_ingest_superstore.py",
    timeout_seconds=300,
    arguments={"catalogo": CATALOGO, "raw_path": f"abfss://raw@{STORAGE_NAME}.dfs.core.windows.net/superstore/"},
)

In [0]:
dbutils.notebook.run(
    f"{BASE_PATH}/02_ingest_ecommerce.py",
    timeout_seconds=300,
    arguments={"catalogo": CATALOGO, "raw_path": f"abfss://raw@{STORAGE_NAME}.dfs.core.windows.net/ecommerce/"},
)

In [0]:
dbutils.notebook.run(
    f"{BASE_PATH}/03_ingest_calendar.py",
    timeout_seconds=300,
    arguments={"catalogo": CATALOGO, "raw_path": f"abfss://raw@{STORAGE_NAME}.dfs.core.windows.net/calendar/"},
)

In [0]:
dbutils.notebook.run(f"{BASE_PATH}/04_transform.py", timeout_seconds=600, arguments={"catalogo": CATALOGO})

In [0]:
dbutils.notebook.run(f"{BASE_PATH}/05_load.py", timeout_seconds=600, arguments={"catalogo": CATALOGO})

In [0]:
dbutils.notebook.run(
    f"{BASE_PATH}/06_grants.py",
    timeout_seconds=300,
    arguments={"sql_file": "seguridad/01_grants.sql", "catalogo": CATALOGO},
)